In [1]:
from discovery_utils.getters import openalex
from discovery_utils import PROJECT_DIR, logging
import pandas as pd
from itertools import combinations, chain
from itertools import product

In [ ]:
# create search terms for variations on "parenting programme"
parent_support_terms = [
    "parenting", "parent", "caregiver"
]

program_keywords = ["intervention", "program", "programme"]

parenting_program_keywords = [f"({p} AND {pr})" for p, pr in product(parent_support_terms, program_keywords)]

combined_parent_keywords = " OR ".join(parenting_program_keywords)

# create search terms for the specified areas of interest
domain_keywords = [
    # Parenting Programmes  which promote children’s early speech, language and communication 
    "speech OR language OR communication",
    # Parenting Programmes which promote children’s early cognitive development
    "cognitive development",
    # Parenting Programmes which promote children’s early personal, social, emotional and behavioral development
    "personal OR social OR emotional",
    # Parenting Programmes which promote children’s physical health and development
    "physical OR health",
    # Programmes which support parents’ confidence, positive behaviours, and skills levels as a parent (including parental self-efficacy)
    "self-efficacy OR confidence OR parenting skills",
    # Programmes which support positive changes in parents beliefs around parenting and parental knowledge  
    "parenting attitudes OR parenting beliefs",
    # Programmes which support parents wellbeing and health
    "parental wellbeing OR maternal wellbeing OR parental health OR maternal health OR parental mental health OR maternal mental health",]

queries = []

for kw in domain_keywords:
    queries.append(f"({combined_parent_keywords}) AND ({kw})")

In [4]:
queries

['((parenting AND intervention) OR (parenting AND program) OR (parenting AND programme) OR (parent AND intervention) OR (parent AND program) OR (parent AND programme) OR (caregiver AND intervention) OR (caregiver AND program) OR (caregiver AND programme)) AND (speech OR language OR communication)',
 '((parenting AND intervention) OR (parenting AND program) OR (parenting AND programme) OR (parent AND intervention) OR (parent AND program) OR (parent AND programme) OR (caregiver AND intervention) OR (caregiver AND program) OR (caregiver AND programme)) AND (cognitive development)',
 '((parenting AND intervention) OR (parenting AND program) OR (parenting AND programme) OR (parent AND intervention) OR (parent AND program) OR (parent AND programme) OR (caregiver AND intervention) OR (caregiver AND program) OR (caregiver AND programme)) AND (personal OR social OR emotional)',
 '((parenting AND intervention) OR (parenting AND program) OR (parenting AND programme) OR (parent AND intervention) O

In [ ]:
data_dfs = []

index = 0

for query in queries:
    logging.info(f"Querying OpenAlex for '{query}'")
    data_df = (
        openalex.get_openalex_works(query, n_works = 10000)
        .assign(query=query)
    )
    data_df.to_csv(f'query_{index}.csv', index=False)
    index += 1
    logging.info(f"Collected {len(data_df)} works")
    data_dfs.append(data_df)
    
data_df = pd.concat(data_dfs, ignore_index=True)

2025-05-15 13:21:25,291 - root - INFO - Querying OpenAlex for '((parenting AND intervention) OR (parenting AND program) OR (parenting AND programme) OR (parent AND intervention) OR (parent AND program) OR (parent AND programme) OR (caregiver AND intervention) OR (caregiver AND program) OR (caregiver AND programme)) AND (speech OR language OR communication)'
2025-05-15 13:22:44,330 - root - INFO - Collected 9502 works
2025-05-15 13:22:44,330 - root - INFO - Querying OpenAlex for '((parenting AND intervention) OR (parenting AND program) OR (parenting AND programme) OR (parent AND intervention) OR (parent AND program) OR (parent AND programme) OR (caregiver AND intervention) OR (caregiver AND program) OR (caregiver AND programme)) AND (cognitive development)'
2025-05-15 13:24:36,400 - root - INFO - Collected 7650 works
2025-05-15 13:24:36,401 - root - INFO - Querying OpenAlex for '((parenting AND intervention) OR (parenting AND program) OR (parenting AND programme) OR (parent AND interven

In [7]:
_data_df = data_df.drop_duplicates("id")
len(_data_df)

40125

In [ ]:
# check difference in size after deduplication
len(data_df) - len(_data_df)

21271

In [ ]:
# manually uploaded to s3://discovery-iss/data/afs_scanning/afs_open_alex_scan_parenting_interventions.csv
_data_df.to_csv("afs_open_alex_scan_parenting_interventions.csv", index=False)